# Phase 3 - FUNSD relation baseline (Colab runner)

Runner only: mount Drive, pull the Phase 3 branch, fetch the raw FUNSD annotations, run the synthetic unit gate, then run the real FUNSD relation evaluation.

Phase 3 is annotation-only and CPU-only. The FUNSD JSON carries entity text, bbox, label, and GT linking pairs, so this notebook does not load image pixels and does not need a GPU. Logic lives in `src/` and `scripts/`, not in this notebook.

Phase 3 is merged to `main`, so the boot cell pins `BRANCH = 'main'`. Repin it to a dev branch only if you resume Phase 3 work.

## Boot

In [ ]:
# 1. Mount Drive so config.DATA_ROOT and config.OUTPUT_ROOT persist across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code onto the VM and pin the Phase 3 branch.
import os

REPO = '/content/FinDocStructRAG'
BRANCH = 'main'  # Phase 3 merged; tracks main

if not os.path.isdir(f'{REPO}/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git {REPO}

!cd {REPO} && git fetch origin --quiet
!cd {REPO} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd {REPO} && echo branch: $(git rev-parse --abbrev-ref HEAD) HEAD: $(git log --oneline -1)
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the Phase 3 paths.
import importlib
import sys

sys.path.insert(0, '/content/FinDocStructRAG')
from src import config
importlib.reload(config)

print('IN_COLAB    :', config.IN_COLAB)
print('DATA_ROOT   :', config.DATA_ROOT)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('FUNSD_ROOT  :', config.FUNSD_ROOT)
print('FUNSD_TRAIN :', config.FUNSD_TRAIN)
print('FUNSD_TEST  :', config.FUNSD_TEST)
print('EVALUATION  :', config.EVALUATION)

## Step 1 - fetch or reuse FUNSD annotations

In [ ]:
# Downloads the official FUNSD zip only if it is not already present on Drive.
# It extracts to data/raw/funsd/dataset/...; tests never depend on this data.
!python scripts/fetch_funsd.py

In [ ]:
# Dataset count sanity check. Expected: 149 train + 50 test annotations.
import importlib
from src import config
importlib.reload(config)

n_train = len(list(config.FUNSD_TRAIN.glob('*.json')))
n_test = len(list(config.FUNSD_TEST.glob('*.json')))
print('train annotations:', n_train)
print('test annotations :', n_test)
assert n_train == 149 and n_test == 50, 'Unexpected FUNSD annotation counts'

## Step 2 - unit gate

In [ ]:
# Small dependency for the synthetic acceptance tests. The Phase 3 runtime itself is stdlib-only.
!python -m pip install -q pytest
!python -m pytest tests/test_funsd_relations.py

## Step 3 - run Phase 3 evaluation

In [ ]:
!python scripts/evaluate_funsd.py

In [ ]:
# Load the JSON report and display the split x scope matrix.
import json
from pathlib import Path

from src import config

report_path = config.EVALUATION / 'phase3_funsd_relations.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('report:', report_path)
print('primary:', report['primary'])
print('note   :', report['note'])

rows = []
for split, scopes in report['results'].items():
    for scope, m in scopes.items():
        rows.append({
            'split': split,
            'scope': scope,
            'precision': round(m['precision'], 3),
            'recall': round(m['recall'], 3),
            'f1': round(m['f1'], 3),
            'tp': m['tp'],
            'pred': m['n_pred'],
            'gold': m['n_gold'],
            'forms': m['num_forms'],
        })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)

## Step 4a - Error table

Notebook-only qualitative error analysis. This reads FUNSD JSON annotations and relation predictions; it does not write artifacts and does not affect the Phase 3 acceptance gate.

In [ ]:
# Build a held-out error table with missed/spurious QA links and entity text/bboxes.
from src.funsd_extraction import load_funsd_split, predict_qa_links, qa_gold_links
from src import config

def _ent_lookup(form):
    return {e['id']: e for e in form['entities']}

def _short(text, n=70):
    text = ' '.join(str(text).split())
    return text if len(text) <= n else text[:n - 1] + '...'

def _link_text(pair, by_id):
    qid, aid = pair
    q = by_id[qid]
    a = by_id[aid]
    return f"{qid}:{_short(q['text'])} -> {aid}:{_short(a['text'])}"

def _link_box(pair, by_id):
    qid, aid = pair
    return {
        'question_id': qid,
        'question_box': by_id[qid]['box'],
        'answer_id': aid,
        'answer_box': by_id[aid]['box'],
    }

test_forms = load_funsd_split(config.FUNSD_TEST)
ERROR_FORMS = {}
ERROR_ROWS = []
ERROR_LINK_ROWS = []

for form in test_forms:
    by_id = _ent_lookup(form)
    pred = predict_qa_links(form)
    gold = qa_gold_links(form)
    correct = pred & gold
    missed = gold - pred
    spurious = pred - gold
    tp = len(correct)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold) if gold else 0.0
    ERROR_FORMS[form['form_id']] = {
        'form': form,
        'pred': pred,
        'gold': gold,
        'correct': correct,
        'missed': missed,
        'spurious': spurious,
        'precision': precision,
        'recall': recall,
    }
    ERROR_ROWS.append({
        'form_id': form['form_id'],
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'tp': tp,
        'pred': len(pred),
        'gold': len(gold),
        'missed': len(missed),
        'spurious': len(spurious),
        'missed_links': '; '.join(_link_text(pair, by_id) for pair in sorted(missed))[:500],
        'spurious_links': '; '.join(_link_text(pair, by_id) for pair in sorted(spurious))[:500],
    })
    for kind, pairs in [('missed', missed), ('spurious', spurious), ('correct', correct)]:
        for pair in sorted(pairs):
            boxes = _link_box(pair, by_id)
            ERROR_LINK_ROWS.append({
                'form_id': form['form_id'],
                'kind': kind,
                'link': _link_text(pair, by_id),
                **boxes,
            })

ERROR_ROWS = sorted(ERROR_ROWS, key=lambda r: (r['recall'], r['precision'], r['form_id']))
WORST_FORM_IDS = [r['form_id'] for r in ERROR_ROWS[:10]]
print('worst form ids:', WORST_FORM_IDS)

try:
    import pandas as pd
    display(pd.DataFrame(ERROR_ROWS[:10]))
    display(pd.DataFrame(ERROR_LINK_ROWS).query("form_id in @WORST_FORM_IDS and kind != 'correct'"))
except Exception:
    for row in ERROR_ROWS[:10]:
        print(row)


## Step 4b - Visualize one error case

Set `FORM_ID` to one held-out form id. The overlay is for human debugging only: missed gold links are red, spurious predicted links are orange, and correct links are green. Question boxes are blue and answer boxes are purple.

In [ ]:
# Visualize one held-out error case with the raw FUNSD image and relation overlay.
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

FORM_ID = WORST_FORM_IDS[0] if WORST_FORM_IDS else None  # or set manually, e.g. '82092117'

def _center(box):
    x0, y0, x1, y1 = box
    return ((x0 + x1) / 2, (y0 + y1) / 2)

def _draw_entity(draw, entity, outline):
    box = entity['box']
    draw.rectangle(box, outline=outline, width=2)
    label = f"{entity['id']} {entity['label']} {_short(entity['text'], 28)}"
    x0, y0, *_ = box
    draw.text((x0, max(0, y0 - 12)), label, fill=outline)

def render_funsd_overlay(form_id, max_links_per_kind=40):
    if form_id not in ERROR_FORMS:
        raise ValueError(f"unknown test form id: {form_id!r}")
    info = ERROR_FORMS[form_id]
    form = info['form']
    by_id = _ent_lookup(form)
    img_path = config.FUNSD_ROOT / 'testing_data' / 'images' / f"{form_id}.png"
    if not img_path.exists():
        raise FileNotFoundError(f"image not found: {img_path}")

    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    colors = {
        'correct': (32, 160, 80),
        'missed': (220, 40, 40),
        'spurious': (245, 140, 30),
        'question': (40, 110, 220),
        'answer': (145, 70, 210),
    }

    involved = set()
    for kind in ['correct', 'missed', 'spurious']:
        pairs = list(sorted(info[kind]))[:max_links_per_kind]
        for qid, aid in pairs:
            q = by_id[qid]
            a = by_id[aid]
            involved.update([qid, aid])
            draw.line([_center(q['box']), _center(a['box'])], fill=colors[kind], width=3)

    for eid in involved:
        entity = by_id[eid]
        outline = colors['question'] if entity['label'] == 'question' else colors['answer']
        _draw_entity(draw, entity, outline)

    print(f"{form_id}: precision={info['precision']:.3f} recall={info['recall']:.3f} "
          f"correct={len(info['correct'])} missed={len(info['missed'])} spurious={len(info['spurious'])}")
    print('image:', img_path)
    return img

display(render_funsd_overlay(FORM_ID))


## Step 4c - Batch preview worst N

Shows a small set of worst held-out overlays inline. Keep `N_PREVIEW` small so the notebook stays responsive.

In [ ]:
N_PREVIEW = 3
for form_id in WORST_FORM_IDS[:N_PREVIEW]:
    display(render_funsd_overlay(form_id))
